# Dataset & Sampling Utilities

Sources: [`utils_newbg.py`](../utils_newbg.py), [`sampling.py`](../sampling.py)

## Overview

The dataset pipeline converts individual compound retention times into **ranked pairs** (and optionally **triplets**).

```
Compounds + RTs + Graphs + Features
           │
    RankDataset.__post_init__()
           │
    ┌──────┴────────────────────┐
    │ preprocess_doublets()     │  handle duplicate compounds
    │ _build_triplet_index()    │  sorted RT arrays per dataset
    │ _transform_pairwise()     │  generate all (anchor, positive) pairs
    └──────────────────────────┘
           │
    __getitem__() → (anchor, positive[, negative]), y, weight, is_confl
```

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import inspect
from utils_newbg import RankDataset, check_integrity

## RankDataset — Key Parameters

In [ ]:
# Show all dataclass fields and defaults
import dataclasses
for f in dataclasses.fields(RankDataset):
    default = f.default if f.default is not dataclasses.MISSING else (
              '<factory>' if f.default_factory is not dataclasses.MISSING else '<required>')
    print(f'{f.name:40s} {str(f.type):30s} default={default}')

### Most Important Parameters

| Parameter | Description |
|---|---|
| `x_mols` | Molecular graphs (dmpnn) or SMILES strings |
| `x_extra` | Extra molecular descriptors (e.g., logP) |
| `x_sys` | System features (column HSM/Tanaka, pH) |
| `y` | Retention times |
| `dataset_info` | Dataset ID per compound (for grouping) |
| `epsilon` | RT difference below which pairs get reduced weight |
| `use_pair_weights` | Weight pairs by RT difference (sigmoid) |
| `use_group_weights` | Balance weight across datasets |
| `downsample_groups` | Downsample large datasets to smallest |
| `cluster` | Group datasets by identical system params |
| `pair_step` / `pair_stop` | Control density of pair generation |
| `conflicting_smiles_pairs` | Known conflicting pairs (boosted weight) |
| `no_inter_pairs` | Skip pairs from different datasets |

## Pair Weight Function

In [ ]:
import matplotlib.pyplot as plt

def weight_fn(x, steep=20, mid=0.75):
    """Sigmoid: f(0)→0, f(mid)=0.5, f(2)→1"""
    return 1 / (1 + np.exp(-steep * (x - mid)))

rt_diffs = np.linspace(0, 3, 300)
plt.figure(figsize=(7, 4))
plt.plot(rt_diffs, weight_fn(rt_diffs), label='steep=20, mid=0.75 (default)')
plt.plot(rt_diffs, weight_fn(rt_diffs, steep=10, mid=0.5), label='steep=10, mid=0.5', linestyle='--')
plt.axvline(0.75, color='gray', linestyle=':')
plt.xlabel('RT difference (min)')
plt.ylabel('Pair weight')
plt.title('Pair weight vs. RT difference')
plt.legend()
plt.tight_layout()
plt.show()

## Triplet Sampling in `__getitem__`

In [ ]:
print(inspect.getsource(RankDataset.__getitem__))

### Triplet sampling strategy:
- **Anchor**: compound at index `x1_indices[i]`
- **Positive**: compound from same dataset with `|RT_pos - RT_anchor| < 2.0 min` (close in time)
- **Negative**: compound from same dataset with `5.0 < |RT_neg - RT_anchor| < 20.0 min` (far in time)
- Falls back to pair-only if no suitable positive or negative found (after 10 retries)

## Doublet Handling

In [ ]:
print(inspect.getsource(RankDataset.preprocess_doublets))

Doublets are compounds with the **same SMILES appearing multiple times** in a dataset with **different retention times** (e.g., different isomers that couldn't be disambiguated). Pairs involving doublets whose RT ranges overlap are filtered out to avoid contradictory training signal.

## Data Integrity Check

In [ ]:
print(inspect.getsource(check_integrity))

`check_integrity()` finds **conflicting pairs**: same compound pair, same system settings, but different predicted order. These are provably unpredictable and can be removed with `--clean_data`.

## Weighted Sampling — `sampling.py`

In [ ]:
from sampling import CustomWeightedRandomSampler, calc_sampling_weights

print(inspect.getsource(calc_sampling_weights))

### Sampling modes:

| Mode | Weights based on |
|---|---|
| `pairs` | Number of pairs per dataset/cluster |
| `compounds` | Number of compounds per dataset/cluster |

Weights are `1 / (count / min_count)` so the smallest dataset gets weight 1.0 and larger datasets are downweighted proportionally. Use `--sampling_sqrt_weights` to use `sqrt(count)` to reduce extreme ratios.

In [ ]:
# Illustrate weighting effect
dataset_pair_counts = {'ds_A': 1000, 'ds_B': 5000, 'ds_C': 300}
min_count = min(dataset_pair_counts.values())

print('Linear weights (1 / count):')
for ds, cnt in dataset_pair_counts.items():
    print(f'  {ds}: {1 / (cnt / min_count):.3f}')

print('\nSqrt weights (1 / sqrt(count)):')
import math
min_sqrt = math.sqrt(min_count)
for ds, cnt in dataset_pair_counts.items():
    print(f'  {ds}: {1 / (math.sqrt(cnt) / min_sqrt):.3f}')

## Minimal RankDataset Example

In [ ]:
# Synthetic toy dataset to explore RankDataset without loading real data
n = 20
np.random.seed(42)

# Fake data: 20 compounds, no extra/sys features, 2 datasets
toy_dataset = RankDataset(
    x_mols=np.arange(n),           # placeholder (no real graphs here)
    x_extra=np.zeros((n, 0)),
    x_sys=np.zeros((n, 1)),
    x_ids=[f'smiles_{i}' for i in range(n)],
    y=np.random.uniform(1, 30, n),
    dataset_info=['ds_A'] * 10 + ['ds_B'] * 10,
    void_info={'ds_A': 0.5, 'ds_B': 0.7},
    use_pair_weights=True,
    epsilon=0.5,
    pair_step=1,
    pair_stop=5,
)

print(f'Total pairs: {len(toy_dataset)}')
print(f'Conflicting pairs: {toy_dataset.is_confl.sum()}')
print(f'Weight range: [{toy_dataset.weights.min():.3f}, {toy_dataset.weights.max():.3f}]')